# Figuring out BCPNN

import sys
sys.path.insert(0,'..')

In [ ]:
from vigipy import *
import pandas as pd

Read the data and only take what we need for the contingency table. Only get first few lines bc data is large

# Source - https://stackoverflow.com/a/69888274
# Posted by David Kaftan
# Retrieved 2026-05-25, License - CC BY-SA 4.0

from pyarrow.parquet import ParquetFile
import pyarrow as pa 

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = 1000)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 


ae_df.tail()

# ae_df=pd.read_parquet(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet")
ae_df.head()
#['AE', 'name', 'count'] ('date' is optional for longitudinal models)

drug_adverse = ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")

drug_adverse["AE"] = drug_adverse["reaction_pt"].astype(str)
drug_adverse["name"] = drug_adverse["drug_name"].astype(str)
drug_adverse["count"] = drug_adverse["count"].astype(int)

drug_adverse = drug_adverse[["AE", "name", "count"]]


drug_adverse.head()

Un sacco di problemi con la funzione che crea la contingency table usata in convert, per cui sovrascrivo con una che funzia x me ora:

import numpy as np
import vigipy.utils.data_prep as data_prep

def fixed_compute_contingency(data_frame, product_label, count_label, ae_label, margin_threshold):
    data_cont = pd.pivot_table(
        data_frame,
        values=count_label,
        index=product_label,
        columns=ae_label,
        aggfunc="sum",
        fill_value=0,
    )
    data_cont = data_cont.astype(float)
    data_cont.index = pd.Index(data_cont.index.astype(str).tolist())
    data_cont.columns = pd.Index(data_cont.columns.astype(str).tolist())

    # Usa boolean mask invece di np.where per evitare il problema PyArrow
    row_mask = np.sum(data_cont.values, axis=1) < margin_threshold
    col_mask = np.sum(data_cont.values, axis=0) < margin_threshold
    
    drop_rows = data_cont.index[row_mask]
    drop_cols = data_cont.columns[col_mask]
    data_cont = data_cont.drop(drop_rows)
    data_cont = data_cont.drop(drop_cols, axis=1)
    return data_cont

data_prep.compute_contingency = fixed_compute_contingency

Creo il dataset x benino che vuole lui per convert

drug_adverse = (
    ae_df.groupby(["drug_name", "reaction_pt"])
    .size()
    .reset_index(name="count")
)

drug_adverse = pd.DataFrame({
    "name":  np.array(drug_adverse["drug_name"], dtype=str),
    "AE":    np.array(drug_adverse["reaction_pt"], dtype=str),
    "count": np.array(drug_adverse["count"], dtype=np.int64),
})

data = convert(drug_adverse)

In [ ]:
import pandas as pd
import duckdb
import sys
sys.path.insert(0, "..")  # per trovare il modulo vigipy locale

from vigipy import GPS
from vigipy.utils import Container
from src.contingency_table import build_contingency_table, qc_contingency_table

ct = pd.read_parquet("data/contingency_table.parquet")

def contingency_to_vigipy(ct: pd.DataFrame) -> tuple[Container, int]:
    """
    Converte la contingency table 2x2 nel formato atteso da vigipy GPS.
    """
    df = ct.copy()
    df = df.rename(columns={
        "drug": "product_name",
        "pt":   "ae_name",
        "a":    "events",
    })
    df["product_aes"]          = df["events"] + df["b"]   # a + b
    df["count_across_brands"]  = df["events"] + df["c"]   # a + c

    N = int(df["n"].iloc[0])  # totale unico per tutto il sottoinsieme

    container = Container(params=False)
    container.data = df[["product_name", "ae_name", "events",
                          "product_aes", "count_across_brands"]]
    container.N    = N

    # Matrice di contingenza pivot (drug x PT) — serve per stimare i prior
    container.contingency = df.pivot_table(
        index="product_name",
        columns="ae_name",
        values="events",
        fill_value=0
    )

    return container, N

container, N = contingency_to_vigipy(ct)
print(f"N totale report: {N}")
print(f"Coppie drug-PT : {len(container.data)}")
container.data.head()


Funziona! caccio dentro bcpnn, the article on Iapatinib sets IC>0, lower limit of CI>0, N>=3

In [ ]:
res=bcpnn(container=container, min_events=3, decision_metric="rank", ranking_statistic="quantile")
# figure out cosa devo fare per fare quello che hanno fatto per iapatinib

In [ ]:
res.all_signals

# bisogna trovare i segnali rilevanti quali sono 
